# Knowledge Base + FAISS Index Builder
**Run order:** `generate_kb.py` → this notebook → `agent_pipeline.py`

Builds a 384-dim FAISS flat index over 24 OCT clinical passages (8 classes × 3 angles).  
Evaluates retrieval precision@3. Saves `kb_index.faiss` and `kb_meta.json`.

## 0. Install & Imports

In [ ]:
# !pip install sentence-transformers faiss-cpu

import json
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

KB_PATH    = Path('knowledge_base.json')   # output of generate_kb.py
INDEX_PATH = Path('kb_index.faiss')
META_PATH  = Path('kb_meta.json')

print('Imports OK')

## 1. Load Knowledge Base

In [ ]:
with open(KB_PATH) as f:
    kb = json.load(f)

print(f'Loaded {len(kb)} passages')
print(f'Classes present: {sorted(set(e["label"] for e in kb))}')
print()
for entry in kb[:3]:
    print(f'[{entry["id"]:12s}] ({entry["angle"]})')
    print(f'  {entry["text"][:120]}...')
    print()

## 2. Embed Passages

`all-MiniLM-L6-v2`: 384-dim, ~80MB, fast CPU inference.  
Embeddings L2-normalized so inner product = cosine similarity.

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

texts = [e['text'] for e in kb]
print(f'Embedding {len(texts)} passages...')

embs = embedder.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype('float32')

print(f'Embedding matrix: {embs.shape}')   # (24, 384)
print(f'Norms (should all be 1.0): min={np.linalg.norm(embs, axis=1).min():.4f} max={np.linalg.norm(embs, axis=1).max():.4f}')

## 3. Build FAISS Index

`IndexFlatIP`: exact brute-force inner product search.  
For 24 vectors this is optimal — no approximation needed.

In [ ]:
dim   = embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embs)

print(f'FAISS index: {index.ntotal} vectors, dim={dim}')

# Save index + metadata
faiss.write_index(index, str(INDEX_PATH))

meta = [{'id': e['id'], 'label': e['label'], 'angle': e['angle'], 'text': e['text']}
        for e in kb]
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Saved → {INDEX_PATH}')
print(f'Saved → {META_PATH}')

## 4. Smoke Test — Top-3 per Class

In [ ]:
print('=== Top-3 retrieved passages per Kermany class ===')
for query in ['CNV', 'DME', 'DRUSEN', 'NORMAL', 'AMD', 'CSR', 'MH', 'DR']:
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, 3)
    print(f'\n  Query: {query}')
    for score, idx in zip(scores[0], idxs[0]):
        e = meta[idx]
        print(f'    [{e["id"]:12s}] score={score:.3f} | {e["text"][:70]}...')

## 5. Retrieval Quality — Precision@3

Ground truth: for each class query, the 3 passages from that class are the relevant ones.  
This is a reasonable assumption given our structured corpus (3 passages × 8 classes).  
Adjust `GROUND_TRUTH` if your `knowledge_base.json` uses different ID conventions.

In [ ]:
# Auto-build ground truth from actual KB IDs
from collections import defaultdict
label_to_ids = defaultdict(list)
for e in meta:
    label_to_ids[e['label']].append(e['id'])

print('Ground truth passage IDs per class:')
for label, ids in sorted(label_to_ids.items()):
    print(f'  {label:8s}: {ids}')

print('\n=== Precision@3 ===')
precisions = []
for query, relevant_ids in sorted(label_to_ids.items()):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, idxs = index.search(q_emb, 3)
    retrieved = [meta[i]['id'] for i in idxs[0]]
    hits      = sum(1 for rid in retrieved if rid in relevant_ids)
    p3        = hits / 3
    precisions.append(p3)
    status    = '✓' if p3 == 1.0 else ('~' if p3 > 0 else '✗')
    print(f'  {status} {query:8s}: P@3={p3:.2f}  retrieved={retrieved}')

mean_p3 = np.mean(precisions)
print(f'\n  Mean P@3: {mean_p3:.3f}')

if mean_p3 < 0.8:
    print('  ⚠ Below 0.8 — consider expanding corpus or using a biomedical encoder (MedCPT)')
else:
    print('  ✓ Retrieval quality sufficient for agent grounding')

## 6. Retrieval Function (import this in agent_pipeline.py)

In [ ]:
# This is the exact function used inside agent_pipeline.py
# Keeping it here for reference and testing

def load_retriever(index_path='kb_index.faiss', meta_path='kb_meta.json'):
    """Load FAISS index + metadata. Returns a retrieve() function."""
    _index    = faiss.read_index(str(index_path))
    _embedder = SentenceTransformer('all-MiniLM-L6-v2')
    with open(meta_path) as f:
        _meta = json.load(f)

    def retrieve(finding_label: str, top_k: int = 3) -> dict:
        q = _embedder.encode([finding_label], normalize_embeddings=True).astype('float32')
        scores, idxs = _index.search(q, top_k)
        passages = [
            {'id': _meta[i]['id'], 'angle': _meta[i]['angle'],
             'text': _meta[i]['text'], 'score': float(s)}
            for s, i in zip(scores[0], idxs[0])
        ]
        return {'query': finding_label, 'passages': passages}

    return retrieve


# Test it
retrieve = load_retriever()
result   = retrieve('DME', top_k=3)
print(f'Query: {result["query"]}')
for p in result['passages']:
    print(f'  [{p["id"]}] score={p["score"]:.3f}')
    print(f'  {p["text"][:100]}...')
    print()

## 7. Fix: Zoom Tool with Guards

**Why the zoom tool failed (42% accuracy on crops):**
- Model trained on full 224×224 OCT scans — crops look nothing like training data
- CAM bounding boxes often cover < 5% of image area — tiny context-free patches
- NORMAL class collapsed to 1.6% recall — no discriminative signal in a texture crop

**Three fixes applied below:**
1. Gate: skip zoom if `full_confidence ≥ 0.75`
2. Minimum area: if crop < 15% of image area, return full image prediction instead
3. Blend: `0.6 × full_probs + 0.4 × crop_probs` instead of replacing

In [ ]:
# Fixed zoom tool — paste this into agent_pipeline.py replacing the old zoom()

def zoom_fixed(image_path: str, bbox: list, class_name: str,
               full_probs: dict, full_confidence: float) -> dict:
    """
    Crop the activated region and blend with full-image prediction.

    Guards:
    - Skips if full_confidence >= 0.75 (model already certain)
    - Skips if crop area < 15% of image (crop too small to be informative)
    - Blends crop and full probabilities (0.6 full + 0.4 crop) instead of replacing
    """
    # Guard 1: confidence gate
    if full_confidence >= 0.75:
        return {
            'skipped': True,
            'reason': f'full_confidence={full_confidence:.2f} >= 0.75, zoom not needed',
            'predicted_class': class_name,
            'confidence': full_confidence,
        }

    img    = Image.open(image_path).convert('RGB')
    W0, H0 = img.size

    # Scale bbox from CAM space (7x7) to image space
    cam_size = 7
    x1 = int(bbox[0] / cam_size * W0);  y1 = int(bbox[1] / cam_size * H0)
    x2 = int(bbox[2] / cam_size * W0);  y2 = int(bbox[3] / cam_size * H0)

    # Guard 2: minimum crop area
    crop_area  = (x2 - x1) * (y2 - y1)
    image_area = W0 * H0
    if crop_area / image_area < 0.15:
        return {
            'skipped': True,
            'reason': f'crop area {crop_area/image_area*100:.1f}% < 15%, too small',
            'predicted_class': class_name,
            'confidence': full_confidence,
        }

    # 10% padding
    pad_x = int((x2 - x1) * 0.1);  pad_y = int((y2 - y1) * 0.1)
    x1 = max(0, x1 - pad_x);  y1 = max(0, y1 - pad_y)
    x2 = min(W0, x2 + pad_x); y2 = min(H0, y2 + pad_y)

    crop   = img.crop((x1, y1, x2, y2))
    tensor = val_tf(crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
    crop_probs_arr = torch.softmax(logits, 1)[0].cpu().numpy()

    # Guard 3: blend full + crop probabilities
    full_probs_arr = np.array([full_probs[c] for c in CLASS_NAMES])
    blended        = 0.6 * full_probs_arr + 0.4 * crop_probs_arr
    pred_idx       = int(blended.argmax())

    return {
        'skipped':         False,
        'predicted_class': CLASS_NAMES[pred_idx],
        'confidence':      float(blended[pred_idx]),
        'probabilities':   {c: float(p) for c, p in zip(CLASS_NAMES, blended)},
        'crop_probs':      {c: float(p) for c, p in zip(CLASS_NAMES, crop_probs_arr)},
        'blend_weights':   '0.6 full + 0.4 crop',
        'crop_region':     [x1, y1, x2, y2],
        'crop_area_pct':   float(crop_area / image_area * 100),
    }


print('Fixed zoom tool defined.')
print('\nKey changes vs original:')
print('  1. Skips if confidence >= 0.75')
print('  2. Skips if crop area < 15% of image')
print('  3. Blends full (0.6) + crop (0.4) probabilities instead of replacing')

## 8. Zoom Ablation: Validate the Fix

Re-run the zoom evaluation with the fixed tool on the same 1000 samples.  
Expected: accuracy should recover from 42% to near the full-image 99.7%  
because most high-confidence samples will be gated out.

In [ ]:
# Assumes eval_model, test_ds, val_tf, CLASS_NAMES, DEVICE are loaded
# from the training notebook — run this after loading them

import random
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

N_EVAL = 1000
indices = random.sample(range(len(test_ds)), N_EVAL)

results = []
zoom_skipped = 0
zoom_called  = 0

for idx in tqdm(indices, desc='Zoom fix eval'):
    img_path, true_label = test_ds.samples[idx]

    # Pass 1: full image
    full_result = classify(img_path)
    pred_class  = full_result['predicted_class']
    full_conf   = full_result['confidence']
    full_probs  = full_result['probabilities']

    # Localize
    loc_result  = localize(img_path, pred_class)
    bbox        = loc_result['bbox_pixels']

    # Pass 2: zoom with guards
    zoom_result = zoom_fixed(img_path, bbox, pred_class, full_probs, full_conf)

    if zoom_result['skipped']:
        zoom_skipped += 1
        final_class = pred_class
    else:
        zoom_called += 1
        final_class = zoom_result['predicted_class']

    results.append({
        'true':       CLASS_NAMES.index(test_ds.classes[true_label]),
        'pred':       CLASS_NAMES.index(final_class),
        'zoom_skipped': zoom_result['skipped'],
    })

true_labels = [r['true'] for r in results]
pred_labels = [r['pred'] for r in results]

print(f'Zoom skipped (gated): {zoom_skipped}/{N_EVAL} ({zoom_skipped/N_EVAL*100:.1f}%)')
print(f'Zoom called:          {zoom_called}/{N_EVAL} ({zoom_called/N_EVAL*100:.1f}%)')
print(f'Final accuracy:       {accuracy_score(true_labels, pred_labels)*100:.2f}%')
print(f'(Original broken zoom: 42.30%)')
print()
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES, digits=4))